# Create Datasets with Sampled SciBERT Embeddings

In [ ]:
# Packages
import pandas as pd
from pathlib import Path
import json
import gc
import numpy as np

## Load Sample Ids

In [1]:
# load sample
# /kaggle/input/datasets/lianestrauch/sample-10/sample_tuning_10per.json

# get sample ids
sample_path = Path("/kaggle/input/datasets/lianestrauch/sample-10/sample_tuning_10per.json")

with open(sample_path, "r") as f:
    sample_info = json.load(f)

sample_ids = pd.DataFrame(sample_info["sample"])

## Load scibert Embeddings and Filter for the Sample Ids

In [2]:
# load scibert embeddings
embeddings_path = Path("/kaggle/input/datasets/lianestrauch/scibert-base/bert-base")
chunk_files = sorted((embeddings_path / "chunks").glob("chunk_*.parquet"))

sciBert_embeddings = pd.read_parquet(
        chunk_files,
    )

# get the embeddings of the sample
sample = sciBert_embeddings[sciBert_embeddings.id.isin(set(sample_ids.id))]

print(sample.shape)
sample.head()

(49919, 5)


,id,layer_last,layer_2nd_last,layer_3rd_last,layer_4th_last
26,103374,"[-0.12949544191360474, 0.5158138871192932, -0....","[-0.8452936410903931, 0.3039453327655792, -0.4...","[-0.7977434396743774, 0.6191445589065552, -0.2...","[-0.7106009125709534, 0.7722778916358948, -0.0..."
34,103382,"[0.6077462434768677, -0.19455060362815857, -1....","[0.15738393366336823, 0.028282206505537033, -1...","[0.44764983654022217, 0.23178380727767944, -1....","[0.2680134177207947, 0.5047345161437988, -1.61..."
48,103396,"[0.07194267958402634, 0.655096709728241, -0.59...","[-0.03182201087474823, 0.35526278614997864, -0...","[0.2909817099571228, 0.970551609992981, -0.762...","[0.5095630884170532, 1.216386079788208, -0.646..."
49,103397,"[0.7052226066589355, 0.42838555574417114, -1.3...","[0.5018846392631531, 0.029419496655464172, -1....","[0.28314408659935, 0.40843161940574646, -1.502...","[0.3070419728755951, 0.9133909344673157, -1.50..."
63,103412,"[0.28017592430114746, 0.1830335259437561, -1.6...","[-0.12344702333211899, 0.11046426743268967, -2...","[0.20052850246429443, 0.5701820254325867, -1.7...","[0.043494872748851776, 0.7046018242835999, -1...."


In [3]:
# Free up RAM
del sciBert_embeddings

gc.collect()

0

## Create and Store the Mean Embedding for the Last Four Layers

In [4]:
layer_cols = ['layer_last', 'layer_2nd_last', 'layer_3rd_last', 'layer_4th_last']

# Shape: (n_samples, 4, 768)
embeddings = np.stack([
    np.stack(sample[col].to_numpy())
    for col in layer_cols
], axis=1)

# Mean over the 4 layers
# Shape: (n_samples, 768)
mean_embeddings = embeddings.mean(axis=1)

# Create output DataFrame
result = pd.DataFrame({
    "id": sample["id"].values,
    "mean_embedding": list(mean_embeddings)
})

result.to_parquet(
    "sample_scibert_base_mean_last4.parquet",
    index=False
)

## Store the Last Layer

In [5]:
#create sample for layer 12

# Last layer
sample[["id", "layer_last"]].to_parquet(
    "sample_scibert_base_last.parquet",
    index=False
)

## Store the Second to Last Layer

In [6]:
# create sample for layer 11

# Second to last layer
sample[["id", "layer_2nd_last"]].to_parquet(
    "sample_scibert_base_2nd_last.parquet",
    index=False
)